# Maya Encoding - Quickstart Guide

This notebook demonstrates the two main encoders in the `maya-encoding` library:

1. **VFDEncoder** (Vigesimal Feature Decomposition) - Encodes numeric features using the Maya base-20 number system
2. **MayaCalendarEncoder** - Encodes temporal features using Maya calendar cycles (Tzolk'in, Haab', Long Count)

In [ ]:
import numpy as np
from maya_encoding import VFDEncoder, MayaCalendarEncoder
from maya_encoding import maya_decompose, to_vigesimal, to_bars_dots

## Part 1: Understanding Maya Numbers

The Maya used a **vigesimal (base-20)** number system with three symbols:
- A **dot** (●) = 1
- A **bar** (━) = 5  
- A **shell** (◎) = 0

Each position represents a power of 20 (ones, twenties, four-hundreds, etc.).

In [ ]:
# Convert 347 to vigesimal (base-20)
digits = to_vigesimal(347)
print(f"347 in base-20: {digits}  (LSB first)")
print(f"  = {digits[0]}×1 + {digits[1]}×20 = {digits[0] + digits[1]*20}")

# Decompose each digit into bars and dots
for i, d in enumerate(digits):
    bars, dots = to_bars_dots(d)
    print(f"  Level {i}: digit={d} → bars={bars}, dots={dots}")

In [ ]:
# Full decomposition
info = maya_decompose(347)
print(f"Full decomposition of 347:")
print(f"  Digits:  {info['digits']}")
print(f"  Bars:    {info['bars']}")
print(f"  Dots:    {info['dots']}")
print(f"  Levels:  {info['n_levels']}")

## Part 2: VFDEncoder for Machine Learning

The `VFDEncoder` transforms numeric features into Maya-inspired multi-scale representations,
creating hierarchical features that can capture patterns at different scales.

In [ ]:
# Basic VFD encoding
X = np.array([[0], [7], [20], [100], [347]])

enc = VFDEncoder(n_levels=2, components="full", normalize=False)
X_encoded = enc.fit_transform(X)

print("Original → Encoded (digit, bars, dots per level):")
names = enc.get_feature_names_out()
print(f"Features: {list(names)}")
print()
for i, val in enumerate(X.ravel()):
    print(f"  {val:>4} → {X_encoded[i]}")

In [ ]:
# Use in a scikit-learn pipeline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Synthetic data: y is nonlinear function of x
np.random.seed(42)
X_train = np.random.randint(0, 400, size=(200, 1)).astype(float)
y_train = (X_train[:, 0] % 20) * 3 + np.random.normal(0, 1, 200)

pipe = Pipeline([
    ("vfd", VFDEncoder(n_levels=2, components="full")),
    ("lr", LinearRegression()),
])
pipe.fit(X_train, y_train)
print(f"Pipeline R² score: {pipe.score(X_train, y_train):.4f}")

## Part 3: MayaCalendarEncoder for Temporal Features

The Maya had sophisticated calendar systems with cycles that don't align with
the Gregorian calendar, potentially capturing patterns that standard temporal
encoding methods miss.

In [ ]:
# Encode dates using Maya calendar systems
dates = np.array([
    "2024-01-01",
    "2024-06-15",
    "2024-12-21",  # Winter solstice
    "2012-12-21",  # End of 13th b'ak'tun
])

mce = MayaCalendarEncoder(
    components=["tzolkin", "haab", "long_count"],
    cyclical=True,
)
encoded = mce.fit_transform(dates)

print("Feature names:")
for name in mce.get_feature_names_out():
    print(f"  {name}")
print(f"\nEncoded shape: {encoded.shape}")
print(f"\nFirst date encoding:\n  {encoded[0]}")

In [ ]:
# Explore the calendar components directly
from maya_encoding.core.calendar import (
    gregorian_to_jdn, jdn_to_tzolkin, jdn_to_haab, jdn_to_long_count
)

# The famous date: December 21, 2012
jdn = gregorian_to_jdn(2012, 12, 21)
tzolkin = jdn_to_tzolkin(jdn)
haab = jdn_to_haab(jdn)
lc = jdn_to_long_count(jdn)

print(f"December 21, 2012:")
print(f"  Tzolk'in: {tzolkin[0]} {tzolkin[1]}")
print(f"  Haab':    {haab[1]} {haab[2]}")
print(f"  Long Count: {'.'.join(str(x) for x in reversed(lc))}")

## Part 4: Handling Edge Cases

VFDEncoder handles negative numbers and floats gracefully.

In [ ]:
# Negative numbers with abs_sign strategy
X_neg = np.array([[-10], [0], [10], [-5], [5]])
enc_neg = VFDEncoder(
    n_levels=1, components="lite",
    normalize=False, handle_negative="abs_sign"
)
result = enc_neg.fit_transform(X_neg)
names = enc_neg.get_feature_names_out()
print(f"Features: {list(names)}")
for i, val in enumerate(X_neg.ravel()):
    print(f"  {val:>4} → {result[i]}")

In [ ]:
# Float handling with auto-scaling
X_float = np.array([[1.5], [2.75], [0.1]])
enc_float = VFDEncoder(
    components="lite", normalize=False,
    handle_float="scale"
)
result = enc_float.fit_transform(X_float)
print(f"Auto scale factor: {enc_float.scale_factor_}")
print(f"Encoded: {result}")

## Next Steps

- See `02_vfd_deep_dive.ipynb` for advanced VFD usage and visualization
- See `03_mce_temporal.ipynb` for time series applications
- See `04_benchmark_results.ipynb` for performance comparisons
- Full documentation at https://danielregalado.github.io/maya-encoding